# PD Voice Pipeline on Google Colab

This notebook clones the repository, installs dependencies, and runs the pipeline in Colab.

In [ ]:
# 1) Clone repository and enter repo root (fail-safe)
from pathlib import Path

REPO_URL = "https://github.com/Nishi-BS23/PD-Pipeline.git"
REPO_DIR = "Pipeline-Implementation"

if "<your-user>" in REPO_URL or "<your-repo>" in REPO_URL:
    raise ValueError("Please set REPO_URL to your real GitHub repository URL.")

!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"

candidate = Path(REPO_DIR)
if not candidate.exists():
    raise FileNotFoundError(f"Clone finished but folder not found: {candidate}")

# Primary rule: use the cloned directory directly.
repo_root = candidate.resolve()

# Optional sanity check (no hard fail):
markers = ["run.sh", "comparative_analysis.py", "audio_segmentation.py"]
present = [m for m in markers if (repo_root / m).exists()]
if not present:
    print("Warning: key marker files not found at repo root; continuing anyway.")
    print("This can happen if the repository layout is different.")

%cd {repo_root}
print("Repo root:", repo_root)
!ls -la | sed -n '1,200p'

In [ ]:
# 2) Install dependencies (with fallback)
from pathlib import Path
import torch

req_candidates = [
    Path('Wav2Vec2/requirements.txt'),
    Path('wav2vec2/requirements.txt'),
    Path('requirements.txt'),
]
req_path = next((p for p in req_candidates if p.exists()), None)

!pip -q install --upgrade pip
if req_path is not None:
    print(f"Using requirements file: {req_path}")
    !pip -q install -r "{req_path}" openpyxl umap-learn
else:
    print("requirements.txt not found; installing fallback package list")
    !pip -q install torch torchaudio transformers scipy scikit-learn pandas numpy matplotlib tqdm soundfile openpyxl umap-learn

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3) Provide data files
You need these files/folders inside the repo root:
- `final_selected.xlsx`
- raw audio folder(s) matching `mpower_voice_data_flac*`

If your data is in Google Drive, mount and copy it below.

In [ ]:
# 4) Mount Drive and prepare dataset (use symlink for large raw folder)
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil
import glob
import os

# If your folder name differs, update this one line.
DRIVE_ROOT = Path('/content/drive/MyDrive/PD_DATA')

if not DRIVE_ROOT.exists():
    raise FileNotFoundError(f'DRIVE_ROOT not found: {DRIVE_ROOT}')

print('Drive root:', DRIVE_ROOT)
print('Files/folders under DRIVE_ROOT:')
for p in sorted(DRIVE_ROOT.iterdir()):
    print(' -', p.name)

# Pick metadata file (real file only; not .gsheet)
meta_candidates = []
for pat in ['final_selected.xlsx', 'final_selected.xls', 'final_selected.csv']:
    meta_candidates.extend(glob.glob(str(DRIVE_ROOT / pat)))
meta_candidates = sorted(set(meta_candidates))
if not meta_candidates:
    raise FileNotFoundError(
        'No real metadata file found. Please place final_selected.xlsx or final_selected.csv in DRIVE_ROOT.'
    )
meta_src = Path(meta_candidates[0])

# Pick raw folder (supports mpower_voice_data_flac*)
raw_candidates = sorted(glob.glob(str(DRIVE_ROOT / 'mpower_voice_data_flac*')))
if not raw_candidates:
    raise FileNotFoundError('Could not find mpower_voice_data_flac* folder under DRIVE_ROOT')
raw_src = Path(raw_candidates[0])

# Copy small metadata file
if meta_src.suffix.lower() == '.csv':
    shutil.copy2(meta_src, Path('./final_selected.csv'))
    print('\nCopied metadata:', meta_src, '-> ./final_selected.csv')
else:
    shutil.copy2(meta_src, Path('./final_selected.xlsx'))
    print('\nCopied metadata:', meta_src, '-> ./final_selected.xlsx')

# IMPORTANT: avoid copying huge raw folder from Drive (can fail with I/O error).
# Create symlink in repo root so existing pipeline paths still work.
dst_raw = Path('./') / raw_src.name
if dst_raw.exists() or dst_raw.is_symlink():
    if dst_raw.is_symlink() or dst_raw.is_file():
        dst_raw.unlink()
    else:
        shutil.rmtree(dst_raw)
os.symlink(raw_src, dst_raw, target_is_directory=True)
print('Linked raw folder:', dst_raw, '->', raw_src)

!ls -la | sed -n '1,120p'

In [ ]:
# 5) Run FULL mode (0s-10s segmentation + embeddings + analysis)
from pathlib import Path
import os
import subprocess
import torch

GPU_ARG = '--gpu 0' if torch.cuda.is_available() else ''

# Balanced cap per class: N means PD=N and HC=N across the full flow.
NUM_IMAGES_PER_CLASS = 1000
SEED = 42

# Fallback clone settings (used only if repo root cannot be found).
REPO_URL = 'https://github.com/Nishi-BS23/PD-Pipeline.git'
FALLBACK_REPO_DIR = Path('/content/Pipeline-Implementation')

def safe_exists(p: Path) -> bool:
    try:
        return p.exists()
    except OSError:
        return False

def is_repo_root(p: Path) -> bool:
    return (
        safe_exists(p / 'audio_segmentation.py')
        and safe_exists(p / 'comparative_analysis.py')
        and safe_exists(p / 'Wav2Vec2' / 'pipeline.py')
        and safe_exists(p / 'HuBERT' / 'pipeline.py')
    )

def find_repo_root() -> Path | None:
    # Fast checks first
    candidates = [
        Path.cwd(),
        Path('/content/Pipeline-Implementation'),
        Path('/content/PD-Pipeline'),
        Path('/content/drive/MyDrive/Pipeline-Implementation'),
    ]
    for c in candidates:
        if is_repo_root(c):
            return c

    # Robust fallback: use find command (avoids Python OSError on unstable mount entries).
    cmd = "find /content -maxdepth 6 -type f -name audio_segmentation.py 2>/dev/null | head -n 30"
    out = subprocess.check_output(['bash', '-lc', cmd], text=True)
    for line in out.splitlines():
        p = Path(line.strip()).parent
        if is_repo_root(p):
            return p
    return None

repo_root = find_repo_root()
if repo_root is None:
    print('Repo root not found; cloning fallback repo...')
    if not FALLBACK_REPO_DIR.exists():
        !git clone "$REPO_URL" "$FALLBACK_REPO_DIR"
    repo_root = FALLBACK_REPO_DIR if is_repo_root(FALLBACK_REPO_DIR) else None

if repo_root is None:
    raise FileNotFoundError(
        'Could not locate repo root automatically. Run Cell 1 once and then rerun this cell.'
    )

os.chdir(repo_root)
print('Running from repo root:', repo_root)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

!python audio_segmentation.py --mode full --max-per-class $NUM_IMAGES_PER_CLASS --seed $SEED
!python Wav2Vec2/pipeline.py --mode full $GPU_ARG --max-per-class $NUM_IMAGES_PER_CLASS --seed $SEED
!python HuBERT/pipeline.py --mode full $GPU_ARG --max-per-class $NUM_IMAGES_PER_CLASS --seed $SEED
!python comparative_analysis.py --mode full --n-trials 20 --final-epochs 100 --max-per-class $NUM_IMAGES_PER_CLASS --seed $SEED

In [ ]:
# 6) (Optional) Run SEGMENT mode instead
# NUM_IMAGES_PER_CLASS = 1000
# SEED = 42
# !python audio_segmentation.py --mode segment --max-per-class $NUM_IMAGES_PER_CLASS --seed $SEED
# !python Wav2Vec2/pipeline.py --mode segment $GPU_ARG --max-per-class $NUM_IMAGES_PER_CLASS --seed $SEED
# !python HuBERT/pipeline.py --mode segment $GPU_ARG --max-per-class $NUM_IMAGES_PER_CLASS --seed $SEED
# !python comparative_analysis.py --mode segment --n-trials 20 --final-epochs 100 --max-per-class $NUM_IMAGES_PER_CLASS --seed $SEED

In [ ]:
# 7) Show output artifacts
!find results -maxdepth 4 -type f | head -n 100